In [ ]:
"""
Optional settings: Settings to limit this notebook to only a specific GPU and Huggingface directory
"""
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Here you can select a subset of your GPUs, if you wish to limit prediction only only one graphics card
#os.environ['HF_HOME'] = 'OPTIONAL_PATH_WHERE_HUGGINGFACE_MODELS_SHOULD_BE_DOWNLOADED' # Here you can set a path to which Phi-4 from huggingface should be downloaded to. 

In [ ]:
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm
from evaluate import load

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    
if IN_COLAB:
    os.chdir("/content")
    print("Running in Colab. Changed directory to:", os.getcwd())


### Evaluation
Run the following cells to run the VLMs GPT-5 and/or Phi-4 on your labeled evaluation dataset

In [ ]:
"""

SETTINGS

"""

USE_OPEN_AI = True
OPEN_AI_MODEL = 'gpt-5'
OPENAI_API_KEY = "YOUR_OPEN_AI_API_KEY"

USE_LOCAL_VLM = True

DATASET_PATH = 'Example_Dataset/dataset.csv'
PERFORMANCE_SUMMARY_PATH = './training_summary_vlm.csv'

In [ ]:
def compute_weighted_metrics(y_true: pd.Series, y_pred: pd.Series) -> dict:
    precision_metric = load("precision")
    recall_metric = load("recall")
    f1_metric = load("f1")
    accuracy_metric = load("accuracy")

    y_true = y_true.astype(str)
    y_pred = y_pred.astype(str)

    labels = sorted(y_true.unique().tolist())
    label2id = {label: i for i, label in enumerate(labels)}

    y_true = y_true.map(label2id)
    y_pred = y_pred.map(label2id)

    # Drop rows where prediction is not a known label
    mask = y_pred.notna()
    y_true = y_true[mask].astype(int)
    y_pred = y_pred[mask].astype(int)

    precision = precision_metric.compute(
        predictions=y_pred, references=y_true, average="weighted"
    )["precision"]
    recall = recall_metric.compute(
        predictions=y_pred, references=y_true, average="weighted"
    )["recall"]
    f1 = f1_metric.compute(
        predictions=y_pred, references=y_true, average="weighted"
    )["f1"]
    accuracy = accuracy_metric.compute(
        predictions=y_pred, references=y_true
    )["accuracy"]

    return {
        "accuracy": accuracy,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1,
    }

if USE_OPEN_AI:
    from openai import OpenAI
    import base64
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    client = OpenAI()

    def encode_image(image_path):
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode("utf-8")

    def prepare_examples_gpt(prompt:str, df_examples:pd.DataFrame):
        messages = []
        for i in df_examples.index:  
            base64_image = encode_image(df_examples.at[i, 'image_path'])
            messages.append({
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{base64_image}",
                    },
                ],
            })
            messages.append({
                    "role": "assistant",
                    "content": [
                        {
                            "type": "output_text",
                            "text": f"'{df_examples.at[i, 'label']}'",
                        }
                    ],
                })
        return messages

    def predict_gpt(inference_img_path:str, prompt:str, messages_examples:list):
        messages = messages_examples.copy()
        base64_image = encode_image(inference_img_path)

        messages.append({
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{base64_image}",
                    },
                ],
            })


        response = client.responses.create(
            model=OPEN_AI_MODEL,
            input=messages,
        )
        
        return response.output_text

if USE_LOCAL_VLM:
    from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model_path = "microsoft/Phi-4-multimodal-instruct"
    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path, 
        device_map=device,
        torch_dtype=torch.float16, 
        trust_remote_code=True, 
    ).cuda()

    generation_config = GenerationConfig.from_pretrained(model_path)

    def get_prompt_with_examples_phi(base_prompt:str, df_example: pd.DataFrame):
        prompt = ''
        for i in df_example.index:
            prompt = f"{prompt}<|user|><|image_{i+1}|>{base_prompt}<|end|><|assistant|>'{df_example.at[i, 'label']}'<|end|>"
        prompt = f"{prompt}<|user|><|image_{df_example.shape[0]+1}|>{base_prompt}<|end|><|assistant|>"
        return prompt

    def predict_phi(inference_img_path:str, prompt:str, example_images:list):
        images = example_images.copy()
        image_predict = Image.open(inference_img_path).convert('RGB')
        images.append(image_predict)
    
        inputs = processor(text=prompt, images=images, return_tensors='pt').to('cuda')
        generate_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            generation_config=generation_config,
        )
        generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]
        response = processor.batch_decode(
            generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]
        return response

In [ ]:
""" 

Loading your dataset

"""

df = pd.read_csv(DATASET_PATH)

required_cols = {'image_path', 'label'}
assert required_cols.issubset(df.columns), \
    'Please make sure that your dataset contains both columns: image_path and label.'

df.head()

In [ ]:
classes = list(df.label.unique())
prompt = f'Assign the best fitting class to describe the image by choosing one of the following classes: {classes}'
df_examples = df.groupby('label').apply(lambda x: x.sample(1, random_state=1)).reset_index(drop=True)
df = df[~df["image_path"].isin(df_examples["image_path"])].reset_index(drop=True)
assert not set(df_examples.image_path).intersection(set(df.image_path)), "The example images should not be part of the evaluation dataset."
n_examples = len(df_examples.label.unique())

if USE_OPEN_AI:
    example_messages_gpt = prepare_examples_gpt(prompt, df_examples)
if USE_LOCAL_VLM:
    prompt_phi = get_prompt_with_examples_phi(prompt, df_examples)
    example_images = []
    for index in df_examples.index:
        example_images.append(Image.open(df_examples.at[index, 'image_path']))

for i in tqdm(df.index):
    inference_image_path = df.at[i, 'image_path']
    if USE_OPEN_AI:
        df.at[i, 'Pred_GPT'] = predict_gpt(inference_image_path, prompt, example_messages_gpt)
    if USE_LOCAL_VLM:
         df.at[i, 'Pred_Phi'] = predict_phi(inference_image_path, prompt_phi, example_images)

    if i % 10 == 0:
        df.to_csv('dataset_vlm_predictions.csv', index=False)

if USE_OPEN_AI:
    df['Pred_GPT'] = df['Pred_GPT'].apply(lambda x: x.strip().strip("'\"") if isinstance(x, str) else x)
if USE_LOCAL_VLM:
    df['Pred_Phi'] = df['Pred_Phi'].apply(lambda x: x.strip().strip("'\"") if isinstance(x, str) else x)
df.to_csv('./dataset_vlm_predictions.csv', index=False)

metrics_rows = []
for pred_col in [col for col in df.columns if 'Pred_' in col]:
    if pred_col in df.columns:
        mask = df[pred_col].notna()
        metrics = compute_weighted_metrics(df.loc[mask, "label"], df.loc[mask, pred_col])
        metrics_rows.append({"model": pred_col, **metrics})

df_metrics = pd.DataFrame(metrics_rows)
df_metrics.to_csv('./evaluation_summary_vlm.csv', index=False)

### Inference
Run the following cells to run the VLMs GPT-5 and/or Phi-4 on new unseen data

In [ ]:
import glob
prediction_image_paths = glob.glob('./Example_Dataset/Unseen_Samples/*')
df_predictions = pd.DataFrame(prediction_image_paths, columns=['image_path'])

In [ ]:
best_vlm = 'GPT' if 'GPT' in df_metrics.loc[df_metrics.accuracy.idxmax(), 'model'] else 'Phi'
for i in tqdm(df_predictions.index):
    if best_vlm == 'GPT':
        df_predictions.at[i, 'Pred'] = predict_gpt(df_predictions.at[i, 'image_path'], prompt, example_messages_gpt)
    else:
        df_predictions.at[i, 'Pred'] = predict_phi(df_predictions.at[i, 'image_path'], prompt_phi, example_images)

df_predictions['Pred'] = df_predictions['Pred'].apply(lambda x: x.strip().strip("'\"") if isinstance(x, str) else x)
df_predictions.to_csv('./prediction_unseen_data_vlm.csv', index=False)